# Embeddings-only RAG Test

This notebook tests a simple retrieval pipeline:
- Chunk `html/cleaned_text/*.txt`
- Embed chunks locally with **SentenceTransformers** (`sentence-transformers/all-MiniLM-L6-v2`)
- Build a **FAISS** index offline and cache it to disk
- At runtime: embed each question, retrieve top-k chunks with cosine similarity

**Prereqs**
- You have the embedding model downloaded into your local HF cache (so evaluation can run with no network)
- You have pre-built the embedding lookup index with `python embeddings.py build` or have the following embeddings files downloaded
    - embeddings_cache/embeddings_inly_index.faiss
    - embeddings_cache/embeddings_inly_index.pkl
- Recommended: run with your conda env: `conda run -n cs288-a3 ...`, with `pip install -r requirements.txt` already run in the env


In [1]:
from sentence_transformers import SentenceTransformer

m = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    local_files_only=True,
)
print("Loaded embedding model locally.")
print("Embedding dim:", len(m.encode(["hello"], normalize_embeddings=True, show_progress_bar=False)[0]))

/opt/anaconda3/envs/cs288-a3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10085.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding model locally.
Embedding dim: 384


In [2]:
import embeddings

# Runtime usage: load the prebuilt index once, then embed each question and retrieve.
embeddings.load_index()

queries = [
    "What are the EECS degree requirements?",
    "How do enrollment permission seats work?",
    "What is the workload policy for GSIs?",
    "What is Cake (the storage scheduler) about?",
]

for q in queries:
    print("=" * 80)
    print("Q:", q)
    results = embeddings.search(q, top_k=5)
    for i, (chunk, src) in enumerate(results, 1):
        print(f"\n[{i}] {src}")
        print(chunk[:400].replace("\n", " ") + ("..." if len(chunk) > 400 else ""))


Q: What are the EECS degree requirements?


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 46734.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[1] eecs_berkeley_edu_academics_undergraduate_compare-majors.txt
shaping our world. ### EECS/ECE/CS Program Comparison Chart | **College of Engineering ECE Major**| **College of Engineering EECS Major**| **College of Computing, Data Science, and Society CS Major** ---|---|---|--- **Admissions**| Admitted directly to the major. (Students must apply directly to ECE at admission if interested)| Admitted directly to the major. (Students must apply directly to EECS ...

[2] eecs_berkeley_edu_academics_undergraduate_eecs-cs-comparison-chart.txt
shaping our world. ### EECS/ECE/CS Program Comparison Chart | **College of Engineering ECE Major**| **College of Engineering EECS Major**| **College of Computing, Data Science, and Society CS Major** ---|---|---|--- **Admissions**| Admitted directly to the major. (Students must apply directly to ECE at admission if interested)| Admitted directly to the major. (Students must apply directly to EECS ...

[3] eecs_berkeley_edu_academics_graduate_industry